# Lesson 5b — Statistics and Uncertainty — Solutions

> **Teacher reference** — This notebook contains worked solutions. Do not share with students.


One of three independent Lesson 5 modules — **5a fitting · 5b statistics · 5c automation** — do them in any order.

This notebook introduces **basic statistical inference**: mean, standard deviation, standard error of the mean, error bars, and a two-sample t-test to decide whether an observed difference between groups is meaningful.

In [ ]:
# Setup cell: run this once at the beginning.
# It installs SciPy (used in this notebook). This is plumbing — just run it.
%pip install -q scipy
print("Setup complete.")

## Statistics and uncertainty

A very common scientific question:

> The quenched samples *look* harder than the annealed samples. Is the difference **meaningful**, or could it just be noise?

Using repeated hardness measurements, we introduce: mean, standard deviation, standard error of the mean, error bars, and a two-sample t-test.

In [ ]:
# Part B — imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

def load_table(name):
    """Robust CSV loader: works locally and in the browser (JupyterLite).
    Do not worry about the details — this is just plumbing."""
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(f"{folder}/{name}")
        except Exception:
            pass
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))

hardness = load_table("hardness_positions.csv")

# Keep only the measurements flagged as valid (same data-quality step as Lesson 4)
hardness = hardness[hardness["is_valid"]]
print(hardness.shape)
hardness.head()

## Standard deviation vs standard error of the mean

The **standard deviation** describes the spread of the individual measurements.

The **standard error of the mean** describes the uncertainty on the *estimated mean*:

$$
SEM = \frac{\text{std}}{\sqrt{n}}
$$

They answer different questions:

```text
std  ->  how scattered are the individual measurements?
SEM  ->  how uncertain is the mean we computed?
```

In [ ]:
# Summary statistics by treatment
summary = (
    hardness
    .groupby("treatment")["hardness_HV"]
    .agg(mean="mean", std="std", n="count")
    .reset_index()
)
summary["sem"] = summary["std"] / np.sqrt(summary["n"])
summary

In [ ]:
# Bar chart: mean hardness with mean +/- SEM error bars
plt.figure(figsize=(6, 4))
plt.bar(summary["treatment"], summary["mean"], yerr=summary["sem"], capsize=5)
plt.ylabel("Hardness [HV]")
plt.xlabel("Treatment")
plt.title("Hardness by treatment: mean +/- SEM")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Two-sample t-test: is quenched really harder than annealed?
annealed = hardness.loc[hardness["treatment"] == "annealed", "hardness_HV"]
quenched = hardness.loc[hardness["treatment"] == "quenched", "hardness_HV"]

# Welch's t-test: does not assume the two groups have equal variance
result = stats.ttest_ind(quenched, annealed, equal_var=False)

print("Two-sample t-test: quenched vs annealed")
print(f"  mean quenched = {quenched.mean():.1f} HV   mean annealed = {annealed.mean():.1f} HV")
print(f"  t statistic   = {result.statistic:.3f}")
print(f"  p-value       = {result.pvalue:.3e}")

if result.pvalue < 0.05:
    print("  Conclusion: the difference is statistically significant at the 0.05 level.")
else:
    print("  Conclusion: not enough evidence for a difference at the 0.05 level.")

### Exercise 1 — Compare another pair of treatments

The t-test above compared quenched vs annealed. Now compare a different pair — e.g. `aged` vs `as_received`.

1. take the `hardness_HV` values for each of the two treatments;
2. run a Welch t-test with `stats.ttest_ind(..., equal_var=False)`;
3. print the p-value.

**Check:** which treatment has the higher mean hardness, and is the difference significant at the 0.05 level? One sentence.

In [ ]:
# Exercise 1 — SOLUTION

t1, t2 = "aged", "as_received"
group1 = hardness.loc[hardness["treatment"] == t1, "hardness_HV"]
group2 = hardness.loc[hardness["treatment"] == t2, "hardness_HV"]

result = stats.ttest_ind(group1, group2, equal_var=False)

print(f"t-test: {t1} vs {t2}")
print(f"  mean {t1}:       {group1.mean():.1f} HV")
print(f"  mean {t2}: {group2.mean():.1f} HV")
print(f"  p-value:              {result.pvalue:.3e}")

if result.pvalue < 0.05:
    print("  Conclusion: the difference is statistically significant at the 0.05 level.")
else:
    print("  Conclusion: not enough evidence for a difference at the 0.05 level.")


### Exercise 2 — Standard error by hand

The summary table computed the SEM for you. Reproduce it yourself for one treatment to see where it comes from.

For the `annealed` group:

1. get its `hardness_HV` values;
2. compute `n` (the number of measurements), the standard deviation, and `SEM = std / sqrt(n)`;
3. print your SEM.

**Check:** does it match the `sem` value for `annealed` in the `summary` table above?

In [ ]:
# Exercise 2 — SOLUTION

annealed_hv = hardness.loc[hardness["treatment"] == "annealed", "hardness_HV"]
n = len(annealed_hv)
s = annealed_hv.std()
sem_manual = s / np.sqrt(n)

print(f"n   = {n}")
print(f"std = {s:.4f} HV")
print(f"SEM = {sem_manual:.4f} HV")

# Compare with the summary table
sem_from_table = summary.loc[summary["treatment"] == "annealed", "sem"].values[0]
print()
print(f"SEM from summary table: {sem_from_table:.4f} HV")
# They match — the formula SEM = std / sqrt(n) is what the table computed.

## Closing — Course recap

This final lesson ties together the whole course:

```text
L1  ->  Foundations & first look
        variables, types, lists, conditions, loops, functions; a read->compute->plot demo

L2  ->  Python basics (hands-on)
        arithmetic, booleans, if/elif, lists, for, functions, dictionaries, basic plots

L3  ->  From raw files to clean data tables
        read_csv, inspect, filter, calculated columns, cleaning, groupby, save

L4  ->  Plotting and data analysis
        line/scatter/hist/box/bar plots, summary metrics, a linear fit

L5  ->  Three practical tools
        nonlinear fitting, uncertainty & statistics, automation over many files
```

The central message:

```text
Python pays off when the analysis must be repeated, checked, modified, or shared.
```

## Where to go next

Three useful directions after this introductory course:

## 1. SciPy
Fitting, optimization, statistics, interpolation, integration, signal processing.
Start with: `scipy.optimize.curve_fit`, `scipy.stats`, interpolation.

## 2. pandas
Tabular data analysis.
Start with: missing values, merging tables, grouping/aggregation, reshaping.

## 3. Matplotlib
Publication-quality figures.
Start with: `fig, ax = plt.subplots()`, axis customization, saving vector graphics, multi-panel figures.

Final practical advice:

```text
Start with small notebooks that solve real problems from your own research.
```